# Task 1: baseline geocoder and evaluation protocol

The source data is synthetic. The full 100-address baseline is descriptive. The 15 surveyed addresses linked to the official account test split have fold 0 and must not guide later model selection. The previous review exposed some test labels, so this is not a pristine blind holdout for the team.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'eval.py').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from eval import baseline_inputs, score_predictions, make_fixed_folds

data = ROOT / 'clean_data'
out_dir = ROOT / 'results'
out_dir.mkdir(exist_ok=True)
predictions, truth, metadata = baseline_inputs(data)
scored, summary = score_predictions(predictions, truth, metadata)
localities = pd.read_csv(data / 'localities.csv')
folds = make_fixed_folds(truth, metadata, localities)
summary.to_csv(out_dir / 'baseline_scores.csv', index=False)
scored.to_csv(out_dir / 'baseline_address_errors.csv', index=False)
folds.to_csv(ROOT / 'folds.csv', index=False)
print(summary[['dimension','segment','n','median_error_m','p90_error_m',
               'share_within_50m','share_within_100m','share_within_250m']].to_string(index=False))


In [ ]:
def median_plot(dimension, title, filename):
    part = summary[summary.dimension == dimension].sort_values('median_error_m')
    fig, ax = plt.subplots(figsize=(9, 4.5))
    x = np.arange(len(part))
    ax.bar(x, part.median_error_m, color='#2563eb')
    for i, row in enumerate(part.itertuples()):
        if pd.notna(row.median_ci95_low_m):
            ax.errorbar(i, row.median_error_m,
                        yerr=[[row.median_error_m-row.median_ci95_low_m],
                              [row.median_ci95_high_m-row.median_error_m]],
                        fmt='none', ecolor='#111827', capsize=3)
    ax.set_xticks(x, [f'{r.segment} (n={r.n})' for r in part.itertuples()],
                  rotation=20, ha='right')
    ax.set(ylabel='Median error (m)', title=title)
    ax.grid(axis='y', alpha=.2)
    fig.tight_layout()
    fig.savefig(out_dir / filename, dpi=160)
    plt.show()

median_plot('precision', 'Baseline error by address precision (95% interval)',
            'baseline_by_precision.png')
median_plot('town_id', 'Baseline error by town (95% interval)',
            'baseline_by_town.png')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
axes[0].hist(scored.error_m, bins=18, color='#0f766e', edgecolor='white')
axes[0].axvline(scored.error_m.median(), color='#b91c1c', linestyle='--',
                label=f'Median {scored.error_m.median():.1f} m')
axes[0].set(xlabel='Error (m)', ylabel='Surveyed addresses', title='Baseline errors')
axes[0].legend()
overall = summary.query("dimension == 'overall'").iloc[0]
axes[1].bar(['50 m','100 m','250 m'],
            [overall.share_within_50m, overall.share_within_100m,
             overall.share_within_250m], color=['#0f766e','#2563eb','#7c3aed'])
axes[1].set(ylim=(0, 1), ylabel='Share of surveyed addresses',
            title='Baseline pins within a radius')
for ax in axes:
    ax.grid(axis='y', alpha=.2)
fig.tight_layout()
fig.savefig(out_dir / 'baseline_error_distribution.png', dpi=160)
plt.show()


In [ ]:
print('Official holdout and grouped development folds:')
print(folds.groupby(['role','fold']).size().to_string())
print('A proxy locality never crosses two development folds:',
      (folds[folds.fold > 0].groupby('fold_group').fold.nunique() == 1).all())
print('Visited and unvisited baseline:')
print(summary[summary.dimension == 'visit_status'][
    ['segment','n','median_error_m','p90_error_m']].to_string(index=False))


## Reading the result

Median error is 376.4 m and P90 is 839.2 m on all 100 surveyed addresses. Only 5%, 9% and 35% lie within 50, 100 and 250 m. Visit status is a descriptive grouping, not an estimate of visit benefit. Confidence intervals resample addresses; intervals for fewer than 10 rows are suppressed. The proxy fold locality comes from the baseline pin's nearest named locality centroid, so spatial leakage may remain across neighboring locality boundaries.
